In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Importing and dataset loading

In [2]:
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

import torch
from torch.nn.functional import softmax

from datasets import Dataset

from transformers import (AutoTokenizer, AutoModelForMultipleChoice, Trainer, TrainingArguments)

from peft import (LoraConfig, TaskType, get_peft_model)

In [3]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_submission = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

test_cc = test.copy()
train_cc = train.copy()
sample_submission_cc = sample_submission.copy()

In [4]:
print("Test info", test.shape)
print("Train info", train.shape)
print("Test_cc shape", test_cc.shape)
print("Train_cc shape", train_cc.shape)
print("Sample_submission.shape", sample_submission.shape)
print("Sample_submission_cc.shape", sample_submission_cc.shape)

Test info (500, 7)
Train info (2000, 8)
Test_cc shape (500, 7)
Train_cc shape (2000, 8)
Sample_submission.shape (500, 2)
Sample_submission_cc.shape (500, 2)


# Label Encoding

Q1. Convert the answer column in train.csv into numeric labels using the following mapping:  A = 0, B =1, C = 2, D = 3, E = 4
what is the encoded numeric label for the row at index 150?



In [5]:
#label mapping
label_mapping = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
#encoding  answer column
train_cc["label"] = train_cc["answer"].map(label_mapping)
#displaying row at index 150
train_cc.loc[150, ["answer", "label"]]

answer    C
label     2
Name: 150, dtype: object

#  Prompt option formatting 

Q2.for now row index 0, create the option B input using exactly this format str(prompt) + *[SEP]* + str(option_B)
what is the exact character length of this formatted input string ??



In [6]:
row = train_cc.loc[0]
option_b_input = str(row["prompt"]) + "[SEP]" + str(row["B"])
print(option_b_input)

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.[SEP]Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.


In [7]:
len(option_b_input)

405

## Tokenization for multi choice models 

Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length

Since each question has five options, every row becomes five tokenized sequences.

## Single Row MCQ

Q3. Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?

In [8]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
row = train_cc.loc[0]
choices = [
    str(row["prompt"]) + "[SEP]" + str(row["A"]),
    str(row["prompt"]) + "[SEP]" + str(row["B"]),
    str(row["prompt"]) + "[SEP]" + str(row["C"]),
    str(row["prompt"]) + "[SEP]" + str(row["D"]),
    str(row["prompt"]) + "[SEP]" + str(row["E"])
]
encoding = tokenizer(choices, padding = "max_length", truncation = True, max_length = 128, return_tensors = "pt")
encoding = {
    k: v.unsqueeze(0)
    for k, v in encoding.items()
}
print(encoding["input_ids"].shape)

torch.Size([1, 5, 128])


# Batch MCQ Tokenization

## Tokenize the first 16 rows of train.csv as multiple-choice examples.

Q4. Each row has 5 choices.
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?



In [10]:
tokenized_batch = []
for _, row in train_cc.head(16).iterrows():
    choices = [
        str(row["prompt"]) + "[SEP]" + str(row["A"]),
        str(row["prompt"]) + "[SEP]" + str(row["B"]),
        str(row["prompt"]) + "[SEP]" + str(row["C"]),
        str(row["prompt"]) + "[SEP]" + str(row["D"]),
        str(row["prompt"]) + "[SEP]" + str(row["E"])
    ]
    choices = tokenizer(
        choices, padding = "max_length",
        truncation = True, max_length = 128, return_tensors = "pt"
    )
    tokenized_batch.append(encoding["input_ids"])

input_ids = torch.stack(tokenized_batch)
print(input_ids.shape)
print(input_ids.numel())

torch.Size([16, 1, 5, 128])
10240


# Multiple-Choice Model Outputs
AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.

## Multiple-Choice Logits

Q5. Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?



In [11]:
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
row = train_cc.loc[0]
choices = [
    str(row["prompt"]) + "[SEP]" + str(row["A"]),
    str(row["prompt"]) + "[SEP]" + str(row["B"]),
    str(row["prompt"]) + "[SEP]" + str(row["C"]),
    str(row["prompt"]) + "[SEP]" + str(row["D"]),
    str(row["prompt"]) + "[SEP]" + str(row["E"]),
]

encoding = tokenizer(choices, padding = "max_length", truncation = True, max_length = 128, return_tensors = "pt")

encoding = {
    k: v.unsqueeze(0)
    for k, v in encoding.items()
}
print(encoding["input_ids"].shape)

with torch.no_grad():
    outputs = model(**encoding)
print(outputs.logits)
print(outputs.logits.shape)

torch.Size([1, 5, 128])
tensor([[-0.8041, -0.7774, -0.7554, -0.7551, -0.7608]])
torch.Size([1, 5])


## Supervised Loss Tensor

Q6.For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?



In [13]:
label = torch.tensor([train_cc.loc[0, "label"]])
print(label)

output = model(
    input_ids = encoding["input_ids"],
    attention_mask = encoding["attention_mask"],
    token_type_ids = encoding["token_type_ids"],
    labels = label
)

print(output.loss)
print(output.loss.shape)
print(output.loss.dim())

tensor([1])
tensor(1.6165, grad_fn=<NllLossBackward0>)
torch.Size([])
0


# LoRA for Efficient Fine-Tuning
LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.


## LoRA Trainable Parameters

Q7. Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?



In [14]:
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
import torch
import transformers
import peft

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

Torch: 2.10.0+cu128
Transformers: 5.0.0
PEFT: 0.19.1


In [16]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 10.2 MB/s eta 0:00:00


In [17]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    r=8, lora_alpha = 16, target_modules = ["query", "value"], lora_dropout = 0.1, bias = "none", task_type = TaskType.SEQ_CLS
)

model = get_peft_model(model, lora_config)
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
print("Trainable Parameters:", trainable_params )

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Trainable Parameters: 295681


# Preparing Data for Hugging Face Trainer
Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.

## Hugging Face Dataset Preparation

Q8.Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:
input_ids with shape [5, 128]
attention_mask with shape [5, 128]
labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?



In [18]:
train_cc_100 = train_cc.head(100).copy()

In [19]:
dataset = []
for _, row in train_cc_100.iterrows():
    choices = [
        str(row["prompt"]) + "[SEP]" + str(row["A"]),
        str(row["prompt"]) + "[SEP]" + str(row["B"]),
        str(row["prompt"]) + "[SEP]" + str(row["C"]),
        str(row["prompt"]) + "[SEP]" + str(row["D"]),
        str(row["prompt"]) + "[SEP]" + str(row["E"]),
    ]
    encoding = tokenizer(choices, padding = "max_length", truncation = True, max_length = 128, return_tensors = "pt")
    dataset.append({
        "input_ids": encoding["input_ids"], "attention_mask": encoding["attention_mask"], "labels": row["label"]
    })

hf_dataset = Dataset.from_list([
    {
        "input_ids": item["input_ids"].tolist(),
        "attention_mask": item["attention_mask"].tolist(),
        "labels": item["labels"]
    }
    for item in dataset
])

print(hf_dataset)
print(np.array(hf_dataset[0]["input_ids"]).shape)
print(np.array(hf_dataset[0]["attention_mask"]).shape)
print(hf_dataset[0]["labels"])

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})
(5, 128)
(5, 128)
1


# Tiny Fine-Tuning and Inference
In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.

## Tiny LoRA Fine-Tuning

Q9. Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4

What is the final global_step reported by the Trainer?

In [20]:
train_cc_32 = train_cc.head(32).copy()

In [21]:
train_dataset = []
for _, row in train_cc_32.iterrows():
    choices = [
        str(row["prompt"]) + "[SEP]" + str(row["A"]),
        str(row["prompt"]) + "[SEP]" + str(row["B"]),
        str(row["prompt"]) + "[SEP]" + str(row["C"]),
        str(row["prompt"]) + "[SEP]" + str(row["D"]),
        str(row["prompt"]) + "[SEP]" + str(row["E"]),
    ]
    encoding = tokenizer(
        choices, padding = "max_length", truncation = True, max_length = 64, return_tensors = "pt"
    )
    train_dataset.append({
        "input_ids": encoding["input_ids"].tolist(),
        "attention_mask": encoding["attention_mask"].tolist(),
        "labels": row["label"]
    })
    hf_train_dataset = Dataset.from_list(train_dataset)
    print(hf_train_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 5
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 6
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 7
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 8
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 9
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 10
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 11
})
Dataset({
    features: ['input_ids', 'attention_mas

In [22]:
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [23]:
# apply LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, lora_config)

In [24]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to="none",
    remove_unused_columns=False
)

In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_train_dataset,
)

trainer.train()

Step,Training Loss
1,3.205311
2,3.249922
3,3.196403
4,3.220003


TrainOutput(global_step=4, training_loss=3.217909574508667, metrics={'train_runtime': 2.9586, 'train_samples_per_second': 10.816, 'train_steps_per_second': 1.352, 'total_flos': 5280340500480.0, 'train_loss': 3.217909574508667, 'epoch': 1.0})

In [26]:
print("Global Step:", trainer.state.global_step)

Global Step: 4


## Probability Assigned to Option E After Fine-Tuning

Q10. Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.

In [27]:
row  = train_cc.loc[0]

choices = [
    str(row["prompt"]) + "[SEP]" + str(row["A"]),
    str(row["prompt"]) + "[SEP]" + str(row["B"]),
    str(row["prompt"]) + "[SEP]" + str(row["C"]),
    str(row["prompt"]) + "[SEP]" + str(row["D"]),
    str(row["prompt"]) + "[SEP]" + str(row["E"]),
]
encoding = tokenizer(
    choices, padding="max_length", truncation=True, max_length=64, return_tensors="pt"
)
device = next(model.parameters()).device

encoding = {k: v.to(device) for k, v in encoding.items()}
encoding = {
    k: v.unsqueeze(0)
    for k, v in encoding.items()
}


In [28]:
model.eval()

with torch.no_grad():
    outputs = model(**encoding)

logits = outputs.logits
print(logits)

tensor([[0.0504, 0.1017, 0.0973, 0.0769, 0.0673]], device='cuda:0')


In [29]:
from torch.nn.functional import softmax

probabilities = softmax(logits, dim=1)

print(probabilities)

option_e_probability = probabilities[0, 4].item()

print("Option E Probability:", option_e_probability)
print("Rounded:", round(option_e_probability, 4))

tensor([[0.1944, 0.2046, 0.2037, 0.1996, 0.1977]], device='cuda:0')
Option E Probability: 0.19768711924552917
Rounded: 0.1977
